<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Relevance judgments

**[Relevance judgments](https://learning.nextia-ai.com/courses/search/m02/relevance-judgments/)** · Search and Ranking: Find the Best Match · Module 2, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read graded relevance judgments, compare two judges in a confusion table, and compute Cohen's kappa by hand.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 8 MB). No account, no key and no model download. |
| **You should know** | The catalogue and the query log, from [Meet the catalogue and the query log](https://learning.nextia-ai.com/courses/search/m01/meet-the-catalogue-and-the-query-log/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m02/relevance-judgments/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M02-L01-relevance-judgments/relevance-judgments-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 10 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/recorded/llm_judgments.jsonl": "f1c7208b12acd187eb46e57def8a0eef3fb7de907dd9620b9000d2ecfce86624",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Graded judgments for one query

A **relevance judgment** says how well one product answers one query. WANDS's judges gave three grades: **exact** (2: what the person asked for), **partial** (1: the right kind of thing, one detail wrong) and **irrelevant** (0). Run the next cell: the judgments of the query "rose gold lounge".

In [ ]:
q = queries().set_index("query_id")
j = judgments()
qid = "W-088"
print(q.loc[qid, "query"], "->", len(j[qid]), "judged products")
print(pd.Series(j[qid]).map({2: "exact", 1: "partial", 0: "irrelevant"}).value_counts())
best = sorted(j[qid], key=lambda p: -j[qid][p])
show(best[:3] + best[-3:], j[qid], n=6)

> **Check.** 150 judged products. Most are partial: WANDS judged many products of the right kind with the wrong colour or style. Read the three best and the three worst: would you grade them the same way?

## 2. Two judges: people and a language model

The course team asked a language model (chat-small) to grade 450 of the same pairs with the course's judging guidelines. Its answers are **recorded**: the notebook reads them, so you need no key. Run the next cell: the table counts the pairs by the WANDS grade (rows) and the model's grade (columns).

In [ ]:
llm = pd.read_json("data/recorded/llm_judgments.jsonl", lines=True)
both = llm[llm["human_grade"] >= 0]          # pairs that the WANDS judges also graded
names = {2: "exact", 1: "partial", 0: "irrelevant"}
table = pd.crosstab(both["human_grade"].map(names).rename("WANDS judges"),
                    both["llm_grade"].map(names).rename("model")).loc[["exact", "partial", "irrelevant"],
                                                                     ["exact", "partial", "irrelevant"]]
agreement = (both["human_grade"] == both["llm_grade"]).mean()
print(f"{len(both)} pairs, raw agreement {agreement:.3f}")
table

> **Check.** 450 pairs and a raw agreement of 0.687: the two judges give the same grade on about two pairs in three.

## 3. Your turn: agreement beyond chance

Raw agreement flatters: two judges who grade at random still agree sometimes. **Cohen's kappa** removes the agreement you would expect by chance: kappa = (observed − chance) / (1 − chance). 1 is perfect, 0 is no better than chance. Complete `cohen_kappa`.

In [ ]:
def cohen_kappa(a, b, labels=(0, 1, 2)):
    """Agreement corrected for the agreement that two judges would reach by chance."""
    a, b = np.asarray(a), np.asarray(b)
    observed = ...   # the share of pairs where a and b are the same
    chance = ...     # the sum over the labels of (share of a with that label) x (share of b with that label)
    if observed is ... or chance is ...:
        return None
    return (observed - chance) / (1 - chance)

kappa = cohen_kappa(both["human_grade"], both["llm_grade"])
print(kappa)

In [ ]:
expect('kappa', kappa, 0.53, tolerance=0.001)

## 4. Where do they disagree?

> **Predict.** When the model and the judges disagree, is the model more often too generous (a higher grade) or too strict?

In [ ]:
too_high = ((both["llm_grade"] > both["human_grade"])).mean()
too_low = ((both["llm_grade"] < both["human_grade"])).mean()
print(f"model grade higher than the judges: {too_high:.3f}; lower: {too_low:.3f}")

> **Check.** Higher on 0.062 of the pairs, lower on 0.251: the model is the **strict** judge. It calls many exact products partial, and many partial products irrelevant. A test set graded only by this model would make every search look worse than it is. The lesson *Judgments from a language model* decides where model labels are good enough.

## 5. In the project

```sh
search judge sample        # 30 results to grade with docs/judging-guidelines.md
search judge agree         # your grades against the WANDS judges: agreement and kappa
search judge llm           # the recorded model labels against the WANDS judges
```

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Ranking measures](https://learning.nextia-ai.com/courses/search/m02/ranking-measures/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m02/relevance-judgments/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.